<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 7C: *Fire Damage Class Balancing*
##### Version Number: 4.0
---
### Contents  
> *Data Preparation*
> *Automatic Class Balancing*
> *Export File*
---
### Notes

**WARNING** this module is computation heavy
- Start with a **baseline model** for comparison.
- Test with multi-classification **tree-based models** (Random Forest, XGBoost) and LGBM.
- Test and evaluate multiple class balancing strategies (No sampling, Oversampling, Undersampling)
- Compare average F1 score among all classes

---
### Inputs
- `X_damage`,`y_damage`,`model_parameters_damage` Reduced or full size Modeling sets

---
### Outputs  

`damage_best_strategy` - dataframe recording best sampling strategy for each method

---
### User Created Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core Python libraries
import numpy as np
import pandas as pd
import json
# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

# Style
sns.set(style='whitegrid')
plt.rcParams["figure.dpi"] = 100

---

In [3]:
X = pd.read_csv('../data/processed/X_damage.csv')
y = pd.read_csv('../data/processed/y_damage.csv').squeeze()

with open('../data/processed/model_parameters_damage.json', 'r') as f:
    model_parameters_damage = json.load(f)

## Prepare Data for Modeling - Scaling, Splitting, and Resampling

In [4]:
damage_rf = RandomForestClassifier(**model_parameters_damage['Random Forest'])
damage_xgb = xgb.XGBClassifier(**model_parameters_damage['XGBoost'])

In [5]:
models = {
    'Damage RF' : damage_rf,
    'Damage XGB' : damage_xgb
}

reform = pd.concat([X,y], axis=1)
subset = balancing_subset_df(reform, 'Target_Damage', 10000,.98,.1,.1)

y = subset['Target_Damage']
X = subset.drop(columns='Target_Damage')

In [6]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

## Automatic Class Balancing

To address the extreme inbalance in the dataset, multiple strategies are explored.
- **In Method Balancing** is used when applicable 
- **RandomUnderSampler** will remove random members of the majority class (Low severity) until they are balanced. This creates a much smaller dataset to model.
- **SMOTE (Synthetic Minority Over-sampling Technique)** will generate synthetic members of the minority classes. Introduces potential noise by synthetic sampling

In [7]:
sampling_strategies = ['Undersampling','No_balance','Oversampling']

all_results = []
counter = 1

for strategy in sampling_strategies:
    for name, model in models.items():
        print("running", strategy, counter, "of 6...")
        counter = counter + 1
        result = class_balancing(
            X_train, y_train, X_test, y_test,
            model, model_parameters_damage,
            sampling_strategy=strategy,
            num_classes=5
        )
        result['Model_Label'] = name
        all_results.append(result)

# Combine into one DataFrame
all_results_df = pd.concat(all_results, axis=0).reset_index(drop=True)

running Undersampling 1 of 6...


running Undersampling 2 of 6...


running No_balance 3 of 6...


running No_balance 4 of 6...


running Oversampling 5 of 6...


running Oversampling 6 of 6...


### Format and display results

In [8]:
strategy_summary = (
    all_results_df[all_results_df['Phase'] == 'Test']
    .groupby(['Balancing','Model_Label'])['F1-Score']
    .mean()
    .reset_index()
    .sort_values(by='F1-Score', ascending=False)
)

strategy_summary_pivot = (
    strategy_summary
    .pivot(index='Model_Label', columns='Balancing', values='F1-Score')
    .sort_values(by='No_balance', ascending=False)
)

strategy_summary_pivot.style.highlight_max(axis=1, color='lightgreen')

Balancing,No_balance,Oversampling,Undersampling
Model_Label,,,
Damage XGB,0.979915,0.974905,0.802444
Damage RF,0.972158,0.964337,0.795487


### Display Best Strategies

In [9]:
# Find best balancing strategy per model
best_strategy = strategy_summary_pivot.idxmax(axis=1)

# Combine with model names into a new DataFrame
best_strategy_df = pd.DataFrame({
    'Model_Label': best_strategy.index,
    'Best_Strategy': best_strategy.values
})

In [10]:
best_strategy_df

,Model_Label,Best_Strategy
0,Damage XGB,No_balance
1,Damage RF,No_balance


### Detailed results (ALL RESULTS)

In [11]:
all_results_df

,Class,Category,Precision,Recall,F1-Score,Phase,Model,Balancing,Model_Label
0,0,0,0.996293,0.872217,0.929991,Train,RandomForestClassifier,Undersampling,Damage RF
1,1,1,0.961570,0.993321,0.977117,Train,RandomForestClassifier,Undersampling,Damage RF
2,2,2,0.962228,0.988125,0.974948,Train,RandomForestClassifier,Undersampling,Damage RF
3,3,3,0.960078,0.990601,0.975063,Train,RandomForestClassifier,Undersampling,Damage RF
4,4,4,0.961936,0.992291,0.976825,Train,RandomForestClassifier,Undersampling,Damage RF
5,0,0,1.000000,0.891135,0.942434,Test,RandomForestClassifier,Undersampling,Damage RF
6,1,1,0.633129,0.994220,0.773613,Test,RandomForestClassifier,Undersampling,Damage RF
7,2,2,0.603816,0.987156,0.749304,Test,RandomForestClassifier,Undersampling,Damage RF
8,3,3,0.626700,0.996071,0.769347,Test,RandomForestClassifier,Undersampling,Damage RF
9,4,4,0.591422,0.998095,0.742736,Test,RandomForestClassifier,Undersampling,Damage RF


---

## Export File

In [12]:
best_strategy_df.to_csv('../data/processed/damage_best_strategy.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
